# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [2]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_74262/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [4]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [5]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [6]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [24]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [8]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

### Homework Experiments:Exploring the PDF
Printing out the number of characters per page to see which pages might contain images based off low character counts. This is not full proof but an interesting test

In this output we can see that indexpages 3-4 (actually pages 2-3) have low character counts. Human eyes show those are primarily tables but other pages do have images

In [12]:
for page in pages:
    print(f"page {page.metadata.get('page')}: {len(page.page_content)} chars")

page 0: 4913 chars
page 1: 2084 chars
page 2: 5955 chars
page 3: 87 chars
page 4: 76 chars
page 5: 5671 chars
page 6: 3588 chars
page 7: 5768 chars
page 8: 5920 chars
page 9: 6015 chars
page 10: 3217 chars
page 11: 5467 chars
page 12: 5698 chars
page 13: 5721 chars
page 14: 6061 chars
page 15: 5360 chars
page 16: 2482 chars
page 17: 5522 chars
page 18: 7086 chars
page 19: 7025 chars
page 20: 7358 chars
page 21: 1514 chars


#### Check for Images
This sections attempts to count the images per page. In this output index pages 3-4 show no images just a low charatcer count. This is interesting because the tables have a high number of characters but the tables don't count as images.

This indicates data may be getting missed when we chunk and embed.

In [13]:
from pypdf import PdfReader

reader = PdfReader(str(pdf_path))
for i, page in enumerate(reader.pages):
    n_images = len(page.images)
    n_chars = len(page.extract_text() or "")
    flag = "  <-- image-only?" if n_images > 0 and n_chars < 50 else ""
    print(f"page {i}: {n_chars} chars, {n_images} images{flag}")

page 0: 4913 chars, 0 images
page 1: 2084 chars, 1 images
page 2: 5955 chars, 0 images
page 3: 87 chars, 0 images
page 4: 76 chars, 0 images
page 5: 5671 chars, 0 images
page 6: 3588 chars, 0 images
page 7: 5768 chars, 0 images
page 8: 5920 chars, 0 images
page 9: 6015 chars, 0 images
page 10: 3217 chars, 1 images
page 11: 5467 chars, 0 images
page 12: 5698 chars, 0 images
page 13: 5721 chars, 0 images
page 14: 6061 chars, 0 images
page 15: 5360 chars, 0 images
page 16: 2482 chars, 0 images
page 17: 5522 chars, 0 images
page 18: 7086 chars, 0 images
page 19: 7025 chars, 0 images
page 20: 7358 chars, 0 images
page 21: 1514 chars, 0 images


#### Try different loaders

Experimenting with other PDF loaders to see how it views these pages

In [15]:
from langchain_community.document_loaders import PDFPlumberLoader
pages = PDFPlumberLoader(str(pdf_path)).load()
print(len(pages[3].page_content), pages[3].page_content[:800])

79 2ELBAT
egatSefiLhcaEgniruDssucsiDromrofrePotsmetI
54 JAAHA | 57:2 Mar/Apr 2021



In [22]:
from langchain_community.document_loaders import PyMuPDFLoader
pages = PyMuPDFLoader(str(pdf_path)).load()
print(len(pages[3].page_content), pages[3].page_content[:800])

87 TABLE 2
Items to Perform or Discuss During Each Life Stage
54
JAAHA
|
57:2
Mar/Apr 2021


#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

The metadata gets embedded into the chunks that can be used to provide transparency of sourcing to the enduser when attached to their query. Depending on what you add as your metadata it can also provide additional context for the searches such as how updated the information might be or who the author was.

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

**Personal Note:** Ran this the first time and it split 22 pages into 133 chunks. I scrolled back and re-ran the original PDFloader to refresh the pages variable and it split 22 pages into 135 chunks.I get consistent results when re-running this after switching loaders again. So its clear that chunks are consistent but only based off the loader and the different loaders can impact you final chunks.

In [37]:
chunk_size = 500
chunk_overlap = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 232 chunks.
Chunk size: 500 characters
Chunk overlap: 50 characters


In [26]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

This is ultimately choosing how much context is within each chunk which impacts the completeness or percision of the returned data. Smaller chunks can be more precise answers but requires retrieving more chunks to build a complete answer. While chunk overlap can help retain context within a chunk but it causes increased storage in duplicate text across chunks

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [39]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [28]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [29]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

Similarity score helps understand the positive relationship between the query and the returned vector chunks. High scores are going to indicate a higher relavance to the query. It is giving a confidence rating but it is not giving validity to the returned data. The data from the vector could be semantically similar but completely incorrect information.

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [30]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [31]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [32]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [33]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that may suggest your cat needs veterinary attention include:

- Pain or anxiety, especially changes in behavior or quality of life [Source 1]
- Changes in grooming, such as increased grooming or reduced grooming [Source 1][Source 2]
- Possible joint or mobility issues, including signs of osteoarthritis or reduced mobility [Source 1][Source 2]
- Abdominal pain or thyroid changes noted during an exam [Source 1]
- Eye changes or signs of high blood pressure detected on a fundic exam [Source 1]
- Vomiting, vomiting hairballs, diarrhea, changes in appetite, or increased thirst/urination [Source 3]
- Increased nocturnal activity, vocalization, or changes in normal habits or activity [Source 3]
- Signs of fear, stress, or distress such as crouching, hiding, flattening ears, dilated pupils, tense body posture, hissing, yowling, growling, or screaming [Source 4]

If you’re seeing any of these, it’s a good idea to contact a veterinarian for an evaluation.

Sources:
{'source_label': 'Sourc

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [34]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guidelines recommend individualized preventive care based on a cat’s life stage, with at least annual veterinary examinations for all cats and more frequent visits as needed for individual risks or chronic conditions. They also note that senior cats should be seen at least every 6 months [Source 4]. Preventive healthcare can include discussion of sterilization, claw care, identification/microchipping, and disaster preparedness [Source 4]. Routine use of broad-spectrum parasite prevention is likely beneficial for most pet cats, and cats with outdoor exposure, travel, or boarding may need additional parasite prevention attention [Source 1].
Question: What symptoms should make me call a veterinarian?
Based on the provided guideline, you should contact a veterinarian if you notice changes in appetite, increased urination or thirst, vomiting, vomiting hairballs, diarrhea, increased nocturnal activity or vocalization, or changes in 

Copy of vibe check that includes outputing context to evaluate for Q4

In [36]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    answer_results = answer_question(question, k=answer_k)
    print("Sources:")
    for source in answer_results["sources"]:
        print(source)
    print("Context: ")
    for context in answer_results["context"]:
        print(context)
    print("Answer: ", answer_results["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1549, 'score': 0.6571027003607005}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 2, 'start_index': 821, 'score': 0.6479922888409305}
{'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 19, 'start_index': 1659, 'score': 0.6267473099907143}
{'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 0, 'score': 0.6201330443648393}
Context: 
(Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 14, 'page_label': '15', 'document_type': 'cat_health_guideline', 'start_index': 1549, '_id': 'a9c11

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

**Vibe Check Query 1:** The first context is very relevant but lacks completeness. It speaks about different preventative care actions but cuts off during the final sentance in reference to recommendations "Thus, recommen- dations for prevention and control should re ﬂect knowledge of the'". The second context is not very relevant is it just talking about cats in the medical system and not specifically about preventative actions. The third chunk provides some preventative care and it is relevant. The final chunk is slightly relevant. It doesn't provide any direct answers but it does provide some external references that could result in more answers if presented to the user that asked the question.

**Vibe Check Query 2:** The first context is very relevant. It calls out specific symptoms or behaviors that should trigger conversations with the vet as well as what questions to ask.The second context has some relevance as it mentions a reason to have a vet visit related to DJD symptoms but mostly the chunk has extra data that isn't relevant. While the third chunk doesn't call out vet calls or visits it does list some symptoms that can be linked to serious conditions which would imply the followup with the Vet. The last chunk is less relevant because it is more about questions and activities during the vet visit and not specifically about reasons to call the vet

**Vibe Check Query 3:** The first chunk as good relevance as it discusses Daily energy requirements (DER) for the cat and how it relates to the food intake. The second context is less direct but does reference food labeled with "Association of American Feed Control" so this would be useful information to provide back to the user. The third chunk includes sections about engery levels, caloric intake and the prevelence of diabities in cat but I don't see it giving any actionable advice and so I would rate it less relevant but adjacent to the question. I would say the last context is also relevant because it discusses the changing in diet as the kitten grows.

**Vibe Check Query 4:** None of these context are relevant because the question is absurd and wouldn't have any relevant data in this PDF.

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [41]:

answer_k = 8
vibe_check_questions = [
    "What preventive care is recommended for cats?",
]

for question in vibe_check_questions:
    print("Question:", question)
    answer_results = answer_question(question, k=answer_k)
    print("Sources:")
    for source in answer_results["sources"]:
        print(source)
    print("Context: ")
    for context in answer_results["context"]:
        print(context)
    print("Answer: ", answer_results["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 1900, 'score': 0.6417932012995435}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1844, 'score': 0.6311731121830666}
{'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 14, 'start_index': 3685, 'score': 0.6307736200695256}
{'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 17, 'start_index': 0, 'score': 0.6265192765342231}
{'source_label': 'Source 5', 'file': 'cat_health_guidelines.pdf', 'page': 16, 'start_index': 2357, 'score': 0.6066231713912486}
{'source_label': 'Source 6', 'file': 'cat_health_guidelines.pdf', 'page': 19, 'start_index': 2226, 'score': 0.6002805296800747}
{'source_label': 'Source 7', 'file': 'cat_health_guidelines.pdf', 'page': 16, 'start_index': 963, 'score': 0.5989882060363104}
{'source_label': 'Source 8', 'file': 'cat_he

### 🏗️ Activity Notes

- Setting changed: Changed chunk size from 1000 to 500 and overlap from 200 to 50
- Before:
    - Split 22 pages into 135 chunks.
    - Chunk size: 1000 characters
    - Chunk overlap: 200 characters
    - Original Sources:
        - {'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1549, 'score': 0.6571027003607005}
        - {'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 2, 'start_index': 821, 'score': 0.6479922888409305}
        - {'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 19, 'start_index': 1659, 'score': 0.6267473099907143}
        - {'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 0, 'score': 0.6201330443648393}
    - Original Answer:  
    The guidelines recommend **regular preventive healthcare** tailored to a cat’s age and individual risk, with **at least annual veterinary examinations for all cats** and **every 6 months for senior cats** or more often if they have chronic conditions [Source 4]. They also note that **routine, regular use of broad-spectrum parasite prevention** is likely beneficial for most pet cats, especially if they have outdoor exposure, travel, or boarding [Source 1].

    Other preventive care topics mentioned include **sterilization, claw care, identification/microchipping, and disaster preparedness** [Source 4]. For parasite control, the guidelines also suggest **treating housemates in synchronicity** when a new kitten or cat is introduced and reducing access to gardens or children’s sand areas to lower parasite exposure [Source 1].

- After:
    - Split 22 pages into 232 chunks.
    - Chunk size: 500 characters
    - Chunk overlap: 50 characters
    - Updated Sources:
       - {'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 1900, 'score': 0.6417931469192264}
       - {'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1844, 'score': 0.6311731121830666}
       - {'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 14, 'start_index': 3685, 'score': 0.6307736200695256}
       - {'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 17, 'start_index': 0, 'score': 0.6265192765342231}
    - Updated Answer:
    Preventive care for cats in the provided guideline includes regular examinations with preventive healthcare and nutritional recommendations, discussing normal behaviors by life stage, and watching for subtle signs of anxiety, illness, and pain so care can be sought early [Source 1]. It also includes regular dental care starting with kitten visits [Source 3], regular cleaning of food and water bowls and considering higher-water-content diets to support hydration [Source 3], and parasite prevention with routine deworming and parasite prophylaxis as appropriate [Source 2][Source 4]. 

    I don't have enough information in the provided cat health guideline PDF to answer that.
- Second set of changes:
    - Kept Chunk settings the same but changed the number of vectors passed in from 4 to 8
    - Second Updated Sources: 
        - {'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 1900, 'score': 0.6417932012995435}
        - {'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1844, 'score': 0.6311731121830666}
        - {'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 14, 'start_index': 3685, 'score': 0.6307736200695256}
        - {'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 17, 'start_index': 0, 'score': 0.6265192765342231}
        - {'source_label': 'Source 5', 'file': 'cat_health_guidelines.pdf', 'page': 16, 'start_index': 2357, 'score': 0.6066231713912486}
        - {'source_label': 'Source 6', 'file': 'cat_health_guidelines.pdf', 'page': 19, 'start_index': 2226, 'score': 0.6002805296800747}
        - {'source_label': 'Source 7', 'file': 'cat_health_guidelines.pdf', 'page': 16, 'start_index': 963, 'score': 0.5989882060363104}
        - {'source_label': 'Source 8', 'file': 'cat_health_guidelines.pdf', 'page': 3, 'start_index': 1362, 'score': 0.5959738671637981}
    - Second Update Answer:  
    Recommended preventive care for cats includes regular preventive healthcare and nutritional recommendations, attention to normal behavior and early signs of anxiety/illness/pain, parasite control, vaccination, regular fecal exams based on health and lifestyle, regular flea prevention, biosecurity measures, and proactive dental care starting with kitten visits [Source 1] [Source 4] [Source 5] [Source 7] [Source 3]. Preventing roaming and limiting access to gardens/play sand areas can also help reduce exposure to zoonotic agents [Source 2] [Source 7].


- Did retrieval improve? Why or why not?

Based of the scores the retrieval almost stayed the same. The two highest scores dropped by a minor amount but the two lower scores actually increased by a similar amount. One might take this to mean we fine tuned to this response to get a tighter more accurate response since the scores got closer together but the overall averaged dropped which means the retrieval quality went down.

When examining the new context the context was still relevant but there were less details in the chunks. Since we had more chunks to choose from but we still only returned 4 vectors to pass to the LLM it had less context to create a complete answer. This results in a partial answer of preventative care with a final comment that it didn't have enough information.

I decided to do another pass with the same chunk settings up increasing the number of vectors passed into the LLM. I was hoping to see if more context produced a more complete or more confident answer. I saw that the top four sources remained consistent but the additional 4 sources showed lower scores. Despite the lower scores the increased context seemd to generate a more complete answer for the user to see and did not respond that it was lacking information